# NB 2 — A minimal agent loop (perceive → plan → act → observe)
**Goal:** make the *control loop* visible. A single agent with a few tools works a recurring clinical task: **post-discharge anticoagulant follow-up.** It reads state, reasons, drafts an action, and stops — it does **not** change any order itself.

No framework is used on purpose — the loop is ~30 lines so you can see exactly what an 'agent' is.

In [1]:

import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (default): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model: pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
USE_MOCK = os.environ.get("OPENAI_API_KEY") is None
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    return r.choices[0].message.content

print("Backend:", "MOCK (no key found — scripted demo)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages):
    """Scripted ReAct actions for the anticoagulant case, driven by which tools
    have already run (visible in the message history)."""
    hist = " ".join(m["content"] for m in messages if m["role"]!="system")
    def done(tool): return f'"observation for {tool}"' in hist or f"[{tool}]" in hist
    if "get_latest_inr" not in hist:
        return json.dumps({"thought":"Start by reading the latest INR.","action":"get_latest_inr","action_input":{}})
    if "get_medications" not in hist:
        return json.dumps({"thought":"INR looks high; check the medication list.","action":"get_medications","action_input":{}})
    if "draft_followup" not in hist:
        return json.dumps({"thought":"INR 4.2 is above the 2-3 range on warfarin. Draft a dose-review flag and earlier follow-up.",
                           "action":"draft_followup","action_input":{"text":"INR 4.2 (above 2-3) on warfarin. Flag for dose review; bring follow-up forward to 3 days."}})
    return json.dumps({"thought":"Handoff drafted for the clinician.","action":"final",
                       "action_input":{"summary":"Drafted a dose-review flag + earlier follow-up. No order changed."}})


Backend: REAL model = openai/gpt-4o-mini


### The environment: tools the agent can use
A tiny synthetic patient. Read-tools return state; the draft-tool proposes text for a human. (Swap these for real FHIR calls later — the loop is unchanged.)

In [2]:
PATIENT = {"inr": 4.2, "range": "2.0-3.0",
           "medications": ["warfarin 5 mg daily", "lisinopril 10 mg daily"],
           "notes": "Discharged 5 days ago after DVT; started on warfarin."}
DRAFTS = []

def _text_arg(a, *names):
    "Pull the intended text no matter which key the model chose for it."
    for n in names:
        if a.get(n): return a[n]
    for v in a.values():
        if isinstance(v, str) and v.strip(): return v
    return ""

# Every tool takes **any keyword. A live model routinely invents or renames
# arguments; a tool that hard-codes its parameter names would crash on them.
def get_latest_inr(**_):    return {"inr": PATIENT["inr"], "reference_range": PATIENT["range"]}
def get_medications(**_):   return {"medications": PATIENT["medications"]}
def get_recent_notes(**_):  return {"notes": PATIENT["notes"]}
def draft_followup(**a):
    text = _text_arg(a, "text","reason","summary","note","content","message")
    DRAFTS.append(text); return {"drafted": text}

TOOLS = {"get_latest_inr":get_latest_inr, "get_medications":get_medications,
         "get_recent_notes":get_recent_notes, "draft_followup":draft_followup}

### The loop
The model returns JSON `{thought, action, action_input}` each turn. We **act** (run the tool), feed the **observation** back, and repeat until it emits `action: "final"`.

In [3]:
SYSTEM = ("You are a clinical follow-up assistant. Each turn, output ONLY JSON: "
          '{"thought":..., "action": one of ' + str(list(TOOLS)+["final"]) + ', "action_input": {{}}}. '
          "Read the INR and medications, and if the INR is out of range, DRAFT a dose-review flag and earlier "
          "follow-up for the clinician. Never change an order yourself. Then action=final.")

def parse(txt):
    "Tolerant: strip code fences, and if the model wrapped prose around it, grab the JSON object."
    t = re.sub(r"^```[a-z]*|```$","",txt.strip(),flags=re.M).strip()
    try:    return json.loads(t)
    except Exception:
        m = re.search(r"\{.*\}", t, re.S)
        return json.loads(m.group(0))

def run(system=SYSTEM, goal="Handle the anticoagulant follow-up.", max_steps=6):
    msgs=[{"role":"system","content":system},{"role":"user","content":goal}]
    for step in range(max_steps):
        act = parse(chat(msgs))
        action = act.get("action","final")
        print(f"\n[{step}] PLAN : {act.get('thought','')}")
        if action=="final":
            print(f"     DONE : {act.get('action_input',{}).get('summary','')}"); return
        tool = TOOLS.get(action)
        if tool is None:
            print(f"     (unknown action {action!r}; stopping)"); return
        obs  = tool(**act.get("action_input",{}))
        print(f"     ACT  : {action}({act.get('action_input',{})})")
        print(f"     OBS  : {obs}")
        msgs.append({"role":"assistant","content":json.dumps(act)})
        msgs.append({"role":"user","content":f"[{action}] observation: {json.dumps(obs)}"})

run()
print("\nDrafts awaiting clinician review:", DRAFTS)


[0] PLAN : I need to check the latest INR and medications to assess the patient's anticoagulant therapy.
     ACT  : get_latest_inr({})
     OBS  : {'inr': 4.2, 'reference_range': '2.0-3.0'}



[1] PLAN : The INR is 4.2, which is out of the therapeutic range (2.0-3.0). I need to draft a dose-review flag and recommend an earlier follow-up for the clinician.
     ACT  : draft_followup({'issue': 'INR out of range', 'current_inr': 4.2, 'reference_range': '2.0-3.0', 'recommendation': 'Consider dose adjustment and earlier follow-up.'})
     OBS  : {'drafted': 'INR out of range'}



[2] PLAN : The follow-up has been successfully drafted regarding the INR being out of range. Now I will finalize the process.
     DONE : 

Drafts awaiting clinician review: ['INR out of range']


### Takeaway
That's the whole idea of an agent: a model **iterating** over perceive→plan→act→observe against an environment. Notice the agent stopped at a *draft* — the consequential action (changing the dose) was never taken. NB 3 makes that boundary explicit with a human gate.